# 06. 과거 센서값으로 파생컬럼 만들기

대상은 `TC-Z3`, `OP-Z3`, `TC-Z4`, `OP-Z4`, `TC-Z5`, `OP-Z5`, `CP-Z4`임. 온도·출력 6개와 CP 1개를 각각 예측하기 위한 **입력 후보**를 준비함. 모델 학습이나 성능 평가는 하지 않음.

이 계산은 기존의 **1분 단위 회귀 입력 후보**를 정리한 것임. 원본 1초 행에 붙일 정상·이상 라벨과 같은 행 단위가 아니고, 최종 분류 모델의 X, y도 아직 아님. 분류에 사용할 때는 판정 시점과 라벨·파생컬럼의 시간 단위를 먼저 맞춰야 함.

**현재 분의 센서 평균이 예측할 값이며, 입력 후보는 현재 분이 시작되기 전에 끝난 분에서만 계산함.** 예를 들어 `13:26` 행의 예측 대상은 `13:26:00 이상 ~ 13:27:00 미만`의 평균임. 이 행의 1분 전 값은 `13:25` 평균이고, 직전 10분 평균은 `13:16~13:25`의 분별 평균을 다시 평균한 값임.

- 모든 시각은 원자료의 실제 날짜·시각이며, 표의 시각은 각 분의 시작임.
- LOT마다 따로 계산함. LOT 사이를 잇거나 관측이 없는 값을 채우지 않음.
- 원자료의 출력 `0`은 유효한 관측으로 유지함. 빈 값과 `0`을 구분하기 위함임.
- 한 분에 필요한 최소 관측 수는 아직 정하지 않음. 기존 임시값인 30개를 강제하지 않고, 센서별 관측 수를 따로 남김.
- [05번 구간 검토](05_interval_review.ipynb)와 같은 공통 함수로 사용자 지정 네 구간을 먼저 제외함. 제외된 센서값은 평균이나 과거값 계산에도 들어가지 않음.
- 모델의 학습·검증·시험 분할이나 정상·이상 정답을 여기서 새로 정하지 않음.

## 1. 데이터가 있는 폴더 찾기
저장소 루트나 이 노트북 폴더에서 실행할 수 있게 하기 위함임. 원자료는 `data/`에서 공통 함수로 읽음.
이 시점 기준은 과거값으로 입력 후보를 만드는 이번 노트북의 약속임. 같은 시각의 다른 센서를 사용하는 팀 모델의 최종 입력 구성을 대신 정한 것은 아님.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# 저장소 루트나 이 노트북 폴더에서 같은 원자료를 읽기 위함임.
ROOT = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data" / "README.md").is_file()
    and (folder / "notebooks").is_dir()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from namelesspark_pipeline import load_sources, build_lot_context, apply_user_exclusions

sources = load_sources(ROOT)
OUTPUT_DIR = ROOT / "data" / "processed" / "features" / "v1"
TARGETS = [
    "TC-Z3", "OP-Z3",
    "TC-Z4", "OP-Z4",
    "TC-Z5", "OP-Z5",
    "CP-Z4",
]
OP_CHANNELS = ["OP-Z3", "OP-Z4", "OP-Z5"]

print("저장소:", ROOT)


## 2. 원본 LOT의 강종을 연결하고 지정 구간 제외하기

LOT의 첫 관측 시각과 CR1 강종 변경 시각이 같은 기록에서 변경 후 강종을 가져옴. 가까운 시각의 강종을 임의로 붙이지 않음. 첫 관측 시각이 실제 설비 가동 시작 시각이라는 뜻은 아님.

LOT의 시작·끝·강종은 **제외 전 원본**에서 연결함. 이후 [05번 구간 검토](05_interval_review.ipynb)와 같은 공통 함수로 사용자 지정 네 구간을 제외함. 061 앞부분을 잘라도 원래 LOT 시작과 강종은 유지되고, 잘린 구간의 값은 아래 1분 평균과 과거 파생컬럼에 들어가지 않음.


In [ ]:
temp_raw = sources["temp"][["MEAS_DT", "LOT_NO"] + TARGETS].sort_values(
    ["LOT_NO", "MEAS_DT"]
)

# 잘린 뒤 첫 관측으로 강종을 다시 연결하지 않도록 원본 문맥을 먼저 남김.
lot_summary = (
    build_lot_context(temp_raw, sources["event"])
    .set_index("LOT_NO")[["START", "END", "ROWS", "GRADE"]]
    .rename(columns={
        "START": "관측시작", "END": "관측끝",
        "ROWS": "원본행수", "GRADE": "강종",
    })
)

# 05와 동일한 네 경계를 적용한 뒤 남은 센서값으로만 계산함.
annotated, exclusion_summary = apply_user_exclusions(temp_raw)
temp = annotated.loc[
    ~annotated["EXCLUDED"], ["MEAS_DT", "LOT_NO"] + TARGETS
].copy()

display(lot_summary)
display(exclusion_summary)
print("원본 행 수:", len(temp_raw))
print("제외 행 수:", int(annotated["EXCLUDED"].sum()))
print("계산에 사용할 행 수:", len(temp))
print("강종이 연결되지 않은 LOT 수:", lot_summary["강종"].isna().sum())


## 3. LOT 안에서 1분 평균과 관측 수 만들기

한 분에 기록된 유효값만 평균함. 센서마다 빈 값이 다를 수 있어 관측 수도 각각 셈. **관측 1개인 평균과 관측 60개인 평균을 같은 품질로 확정하지 않기 위함임.** 지금은 평균과 관측 수를 함께 제공하고, 학습에 필요한 최소 관측 수는 팀에서 정해야 함.

관측이 전혀 없는 분도 시간 위치를 유지함. 평균은 `NaN`, 관측 수는 `0`임. 값이나 선을 보간한 것은 아님.

In [ ]:
minute_frames = []
count_frames = []
saturation_frames = []

for lot_no, lot_data in temp.groupby('LOT_NO', sort=True):
    values = lot_data.set_index('MEAS_DT')[TARGETS]

    # LOT 밖의 자료와 섞이지 않도록 이 LOT에서만 1분씩 묶음.
    minute = values.resample('1min').mean()
    counts = values.resample('1min').count()

    # 빈 OP를 '포화 아님'으로 세지 않기 위해 빈 값은 그대로 남김.
    op_values = values[OP_CHANNELS]
    saturated = op_values.ge(100).where(op_values.notna())
    saturation_count = saturated.resample('1min').sum(min_count=1)

    # 제외 전 원본의 첫 관측이 포함된 분부터 지난 분 수임. 설비 가동 시간은 아님.
    first_minute = lot_summary.loc[lot_no, '관측시작'].floor('min')
    minute['LOT경과_분'] = (minute.index - first_minute).total_seconds() / 60
    minute['LOT_NO'] = lot_no
    minute['강종'] = lot_summary.loc[lot_no, '강종']
    counts['LOT_NO'] = lot_no
    saturation_count['LOT_NO'] = lot_no

    minute_frames.append(minute.reset_index())
    count_frames.append(counts.reset_index())
    saturation_frames.append(saturation_count.reset_index())

minute_data = pd.concat(minute_frames, ignore_index=True)
counts = pd.concat(count_frames, ignore_index=True)
saturation_counts = pd.concat(saturation_frames, ignore_index=True)

# 지정 구간 제외 후 남은 출력 0을 결측으로 바꾸지 않았는지 표시함.
display(temp[OP_CHANNELS].eq(0).sum().rename('제외 후 남은 출력 0 관측 수'))
display(minute_data[['LOT_NO', 'MEAS_DT', '강종'] + TARGETS].head())

### 센서별로 한 분에 몇 개를 평균했는지 확인하기
아래 표는 LOT·센서별 분당 관측 수의 최소·중앙값·최대와, 관측이 없는 분의 수임. 일부 분의 관측 수가 적다는 사실을 숨기지 않기 위함임. 행별 관측 수도 마지막 CSV에 함께 저장함.

In [ ]:
count_table = counts.melt(
    id_vars=['LOT_NO', 'MEAS_DT'],
    value_vars=TARGETS,
    var_name='센서',
    value_name='관측수',
)
count_summary = count_table.groupby(['LOT_NO', '센서'])['관측수'].agg(
    분수='size',
    최소='min',
    중앙값='median',
    최대='max',
    관측없는분=lambda values: values.eq(0).sum(),
).reset_index()

# LOT 순서로 보여 어느 구간에서 관측 수가 적은지 찾기 위함임.
with pd.option_context('display.max_rows', None):
    display(count_summary)

## 4. 현재 분을 빼고 과거값으로만 계산하기

| 새 컬럼 | 계산 |
|---|---|
| `센서_1분전` | 직전 분의 평균 |
| `센서_5분전` | 5분 전의 평균 |
| `센서_직전10분평균` | 직전 10개 분 평균의 평균. 현재 분 제외 |
| `센서_직전10분표준편차` | 직전 10개 분 평균이 얼마나 흔들렸는지. 현재 분 제외 |
| `센서_직전두분차이` | 직전 분 평균 − 2분 전 평균. 시간으로 나눈 변화율이 아님 |
| `OP센서_직전10분포화율` | 직전 10분의 유효 원본 관측 중 출력이 100 이상인 비율(%) |

`shift(1)`은 한 칸 아래로 옮겨 현재 분에서 이전 분의 값을 보게 하는 코드임. `rolling(10, min_periods=10)`은 이전 10개 분의 평균이 모두 있을 때만 계산하기 위함임. 중간에 관측이 없는 분이 있거나 제외 후 남은 구간의 처음 10분이 아직 쌓이지 않았다면 10분 계산은 `NaN`으로 남김. 각 분의 최소 관측 수를 보장한 것은 아님.

포화율은 원본 관측 수를 기준으로 계산함. 관측이 5개인 분과 60개인 분을 똑같은 무게로 평균하지 않기 위함임. 출력 `0`은 유효 관측 수에 포함되지만 포화 관측 수에는 포함되지 않음.

In [ ]:
feature_frames = []

for lot_no, lot_data in minute_data.groupby('LOT_NO', sort=True):
    data = lot_data.set_index('MEAS_DT').copy()
    lot_counts = counts.loc[counts['LOT_NO'].eq(lot_no)].set_index('MEAS_DT')
    lot_saturation = saturation_counts.loc[
        saturation_counts['LOT_NO'].eq(lot_no)
    ].set_index('MEAS_DT')

    for sensor in TARGETS:
        # 현재 예측할 센서값이 입력에 들어가지 않도록 먼저 한 분 옮김.
        past = data[sensor].shift(1)
        data[f'{sensor}_1분전'] = past
        data[f'{sensor}_5분전'] = data[sensor].shift(5)
        data[f'{sensor}_직전10분평균'] = past.rolling(10, min_periods=10).mean()
        data[f'{sensor}_직전10분표준편차'] = past.rolling(10, min_periods=10).std()
        data[f'{sensor}_직전두분차이'] = past.diff()

    for sensor in OP_CHANNELS:
        # 현재 분의 포화 여부도 입력에서 빼기 위함임.
        past_saturation = lot_saturation[sensor].shift(1)
        past_count = lot_counts[sensor].shift(1)
        saturated_sum = past_saturation.rolling(10, min_periods=10).sum()
        valid_sum = past_count.rolling(10, min_periods=10).sum()
        data[f'{sensor}_직전10분포화율'] = saturated_sum / valid_sum * 100

    feature_frames.append(data.reset_index())

features = pd.concat(feature_frames, ignore_index=True)

# 원본 평균과 구분되는 새 파생 컬럼만 입력 후보 목록에 모음.
feature_cols = [column for column in features if column not in minute_data.columns]
input_cols = ['강종', 'LOT경과_분'] + feature_cols

# 관측 수는 품질 확인용임. 예측할 현재 분의 관측 수를 입력으로 넣지 않음.
count_cols = {sensor: f'{sensor}_관측수' for sensor in TARGETS}
features = features.merge(
    counts.rename(columns=count_cols),
    on=['LOT_NO', 'MEAS_DT'],
    how='left',
)

print('예측할 센서:', TARGETS)
print('입력 후보 수:', len(input_cols))
display(features[['LOT_NO', 'MEAS_DT', 'TC-Z3', 'TC-Z3_1분전',
                  'TC-Z3_직전10분평균', 'OP-Z3_직전10분포화율']].head(12))

## 5. 예측할 값과 입력 후보를 나누기

`TARGETS`의 현재 분 평균 7개는 **예측할 값**임. `input_cols`는 강종·관측 시작 후 경과 분과 과거값으로 만든 **입력 후보**임. 각 채널 모델은 예측할 센서를 하나씩 선택하면 됨.

현재 분의 센서값, 현재 분의 관측 수, 현재 값으로 계산한 편차를 입력으로 넣지 않음. 예측할 값을 미리 알려주는 것을 막기 위함임. 입력 후보에 빈 값이 남아 있어도 여기서 `0`으로 채우거나 모든 행을 삭제하지 않음. 모델별 입력 선택과 결측 처리 기준은 학습 단계에서 정해야 함.

| 이전 노트북의 컬럼·설명 | 이번 처리와 이유 |
|---|---|
| `LOT남은_분` | 제외. 관측 종료를 알아야 계산하므로 현재 시점에는 알 수 없는 값임 |
| 전체 기간의 강종 중앙값으로 만든 편차 | 보류. 학습 자료에서만 중앙값을 계산해야 함. 아직 학습 분할을 정하지 않았기 때문임 |
| 전체 기간의 사분위값으로 만든 `정지재가동` | 제외. 전체 기간의 기준값과 단일 센서 조건만으로 정지·재가동을 확정하지 않기 위함임 |
| `TC-Z9_30분진폭` | 이번 표에서는 보류. 여기서는 비교한 7개 센서의 과거값만 준비함. TC-Z9가 입력으로 불필요하다고 확정한 것은 아님 |
| 현재 분을 포함한 이동평균·표준편차·포화율 | 현재 분을 제외한 과거값으로 변경. 예측할 현재 값을 입력으로 넘기지 않기 위함임 |
| `변화율` | `직전두분차이`로 변경. 시간으로 나누지 않은 차이이고, 현재 분도 사용하지 않음 |
| 가스 분석계 셀 교체 후 경과·수명소진율 | 보류. 7개 센서의 과거값을 준비하는 이번 단계와 별도로, 교체 완료 시각과 사용 의미를 정한 뒤 추가할 수 있음 |
| 제외 구간과 라벨 | 현재 05와 동일한 사용자 지정 네 구간은 평균·파생 계산 전에 제외함. 과거 회귀 실험의 보류표나 미확정 라벨을 자동 적용하지 않음 |

한 번만 관측된 강종을 포함해 강종별 정상 기준이 확정된 상태는 아님. 학습 자료에 없던 강종의 처리 방법과 성능 평가는 모델 학습 단계에서 별도로 정해야 함.

In [ ]:
# 모델을 돌리지는 않고 두 표의 역할만 분리해서 확인함.
target_values = features[['LOT_NO', 'MEAS_DT'] + TARGETS]
input_candidates = features[['LOT_NO', 'MEAS_DT'] + input_cols]

print('예측할 값 표:', target_values.shape)
print('입력 후보 표:', input_candidates.shape)
display(pd.Series(input_cols, name='입력 후보 컬럼'))

## 6. 계산한 표를 로컬에 저장하기

`data/processed/features/v1/06_features_1min.csv`에 저장함. 식별용 LOT·시각, 강종, 현재 분의 예측 대상값, 과거값으로 만든 입력 후보, 현재 분의 관측 수가 함께 들어 있음. **CSV 전체를 그대로 모델 입력으로 사용하지 않고 위 `input_cols`로 구분해야 함.**

사용자 지정 네 구간을 제외한 뒤 남은 원본 관측으로 계산한 결과임. 원본 CSV를 덮어쓰지 않음. 계산한 CSV도 Git 커밋 대상이 아님. 시각화 노트북들은 원자료를 직접 읽으므로 이 CSV나 이 노트북의 실행 결과에 의존하지 않음.

03·04번 시각화는 원자료를 직접 읽음. 이 CSV에 1초 행 라벨을 그대로 붙이거나 CSV 전체를 분류 모델에 넣는 단계는 아님.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / '06_features_1min.csv'
features.to_csv(output_path, index=False, encoding='utf-8-sig')
print('저장 위치:', output_path)
print('저장한 행 수:', len(features))